In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import shap
from collections import Counter
from matplotlib import pyplot as plt
from scipy.stats import rankdata

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utilz.Dataset import load_dataset
from utilz.constans import DISEASE, HEALTHY
from utilz.helpers import plot_roc_curve
from utilz.preprocessing_utilz import (
    ConstantExpressionReductor, AnovaReductor,
    MeanExpressionReductor, CovariatesBiasReductor,
)

meta_path = r"../data/samples_pancreatic.xlsx"
data_path = r"../data/counts_pancreatic.csv"
ds = load_dataset(data_path, meta_path, label_col="Group")

TEST_SIZE = 0.2
VALID_SIZE = 0.2
ANOVA_PERCENTILE = 60
MEAN_PERCENTILE = 20

ds.y = ds.y.replace({DISEASE: HEALTHY})

le = LabelEncoder()
y_encoded = pd.Series(le.fit_transform(ds.y), index=ds.y.index)
X_train, X_test, X_valid, y_train, y_test, y_valid = (
    ds.get_train_test_valid_split(ds.X, y_encoded, test_size=TEST_SIZE, valid_size=VALID_SIZE)
)
print("Class mapping:")
print(f"{le.classes_[0]} -> {le.transform(le.classes_)[0]}")
print(f"{le.classes_[1]} -> {le.transform(le.classes_)[1]}")

print("Encoded label distribution:")
print(y_encoded.value_counts().sort_index())
sex_numeric = ds.sex.map({"F": 0, "M": 1})

pipeline = Pipeline([
    ('ConstantExpressionReductor', ConstantExpressionReductor()),
    ('AnovaReductor', AnovaReductor(percentile=ANOVA_PERCENTILE)),
    ('MeanExpressionReductor', MeanExpressionReductor(percentile=MEAN_PERCENTILE)),
    ('AgeBiasReductor', CovariatesBiasReductor(covariate=ds.age)),
    ('SexBiasReductor', CovariatesBiasReductor(covariate=sex_numeric)),
    ('scaler', StandardScaler()),
])

# ── krok 1: ConstantExpressionReductor ──────────────────────────
step1 = ConstantExpressionReductor()
arr = step1.fit_transform(X_train, y_train)

if "ENSG00000105711" in arr.columns:
    print("SCN1B obecny po ConstantExpressionReductor")
else:
    print("SCN1B usunięty po ConstantExpressionReductor")

if "ENSG00000111196" in arr.columns:
    print("MAGOHB obecny po ConstantExpressionReductor")
else:
    print("MAGOHB usunięty po ConstantExpressionReductor")

# ── krok 2: AnovaReductor ────────────────────────────────────────
step2 = AnovaReductor(percentile=ANOVA_PERCENTILE)
arr = step2.fit_transform(arr, y_train)

if "ENSG00000105711" in arr.columns:
    print("SCN1B obecny po AnovaReductor")
else:
    print("SCN1B usunięty po AnovaReductor")

if "ENSG00000111196" in arr.columns:
    print("MAGOHB obecny po AnovaReductor")
else:
    print("MAGOHB usunięty po AnovaReductor")

# ── krok 3: MeanExpressionReductor ──────────────────────────────
step3 = MeanExpressionReductor(percentile=MEAN_PERCENTILE)
arr = step3.fit_transform(arr, y_train)

if "ENSG00000105711" in arr.columns:
    print("SCN1B obecny po MeanExpressionReductor")
else:
    print("SCN1B usunięty po MeanExpressionReductor")

if "ENSG00000111196" in arr.columns:
    print("MAGOHB obecny po MeanExpressionReductor")
else:
    print("MAGOHB usunięty po MeanExpressionReductor")


# ── krok 4: AgeBiasReductor ──────────────────────────────────────
step4 = CovariatesBiasReductor(covariate=ds.age)
arr = step4.fit_transform(arr, y_train)

if "ENSG00000105711" in arr.columns:
    print("SCN1B obecny po AgeBiasReductor")
else:
    print("SCN1B usunięty po AgeBiasReductor")

if "ENSG00000111196" in arr.columns:
    print("MAGOHB obecny po AgeBiasReductor")
else:
    print("MAGOHB usunięty po AgeBiasReductor")

# ── krok 5: SexBiasReductor ──────────────────────────────────────
step5 = CovariatesBiasReductor(covariate=sex_numeric)
arr = step5.fit_transform(arr, y_train)

if "ENSG00000105711" in arr.columns:
    print("SCN1B obecny po SexBiasReductor")
else:
    print("SCN1B usunięty po SexBiasReductor")

if "ENSG00000111196" in arr.columns:
    print("MAGOHB obecny po SexBiasReductor")
else:
    print("MAGOHB usunięty po SexBiasReductor")



[INFO] skipped 1973 probs due to missing metadata
[INFO] 8 samples with unique strata added to train set
[INFO] 4 samples with unique strata (2nd split) added to train set

[ASSERTION PASSED] No leakage detected between splits.
Class mapping:
Asymptomatic controls -> 0
Pancreatic cancer -> 1
Encoded label distribution:
0    460
1    124
Name: count, dtype: int64
data shape after ConstantExpressionReductor:  (357, 30733)
SCN1B obecny po ConstantExpressionReductor
MAGOHB obecny po ConstantExpressionReductor
data shape after AnovaReductor:  (357, 12293)
SCN1B obecny po AnovaReductor
MAGOHB obecny po AnovaReductor
data shape after MeanExpressionReductor:  (357, 9834)
SCN1B obecny po MeanExpressionReductor
MAGOHB obecny po MeanExpressionReductor
data shape after CovariatesBiasReductor:  (357, 7491)
SCN1B usunięty po AgeBiasReductor
MAGOHB usunięty po AgeBiasReductor
data shape after CovariatesBiasReductor:  (357, 7477)
SCN1B usunięty po SexBiasReductor
MAGOHB usunięty po SexBiasReductor


In [8]:
import statsmodels.api as sm

def gene_age_correlation(gene_id: str, X: pd.DataFrame, age: pd.Series, y: pd.Series, p_thresh: float = 0.05):

    # wyrównanie indeksów
    age_s = pd.Series(age).reindex(X.index).astype(float)
    valid_idx = age_s.dropna().index
    age_s = age_s.loc[valid_idx]

    covs = pd.DataFrame(index=valid_idx)
    covs["covariate"] = age_s
    if y is not None:
        y_s = pd.Series(y).reindex(X.index).loc[valid_idx].astype(float)
        covs["disease"] = y_s

    covs_matrix = sm.add_constant(covs)
    cov_col_idx = list(covs_matrix.columns).index("covariate")

    expr = X.loc[valid_idx, gene_id].astype(float).values

    model = sm.OLS(expr, covs_matrix.values).fit()

    p_raw   = model.pvalues[cov_col_idx]
    beta    = model.params[cov_col_idx]
    ci      = model.conf_int()[cov_col_idx]
    t_stat  = model.tvalues[cov_col_idx]
    r2      = model.rsquared

    print(f"Gen:              {gene_id}")
    print(f"Beta (wiek):      {beta:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")
    print(f"t-statistic:      {t_stat:.4f}")
    print(f"p-value (raw):    {p_raw:.4e}")
    print(f"R² modelu:        {r2:.4f}")
    print(f"Istotny (p<{p_thresh}): {'TAK' if p_raw < p_thresh else 'NIE'}")

    return

In [9]:
gene_age_correlation("ENSG00000105711", X_train, ds.age, y_train)
gene_age_correlation("ENSG00000111196", X_train, ds.age, y_train)

Gen:              ENSG00000105711
Beta (wiek):      0.0131  (95% CI: [0.0085, 0.0177])
t-statistic:      5.6078
p-value (raw):    4.2396e-08
R² modelu:        0.1476
Istotny (p<0.05): TAK
Gen:              ENSG00000111196
Beta (wiek):      -0.0073  (95% CI: [-0.0124, -0.0023])
t-statistic:      -2.8559
p-value (raw):    4.5547e-03
R² modelu:        0.1111
Istotny (p<0.05): TAK


# Analiza korelacji ekspresji genów z wiekiem

Metoda: regresja liniowa OLS z uwzględnieniem diagnozy jako kowariatu
(model: `ekspresja ~ const + wiek + diagnoza`).
Istotność oceniano na poziomie α = 0.05, korekcja FDR metodą Benjaminiego-Hochberga.

---

## SCN1B (ENSG00000105711)

Ekspresja genu SCN1B wykazuje istotną dodatnią korelację z wiekiem pacjenta
po uwzględnieniu diagnozy (β = 0.0131, 95% CI: [0.0085, 0.0177], t = 5.608, p = 4.24×10⁻⁸, R² = 0.148).
Gen został wykluczony z dalszej analizy przez `CovariatesBiasReductor`.

---

## MAGOHB (ENSG00000111196)

Ekspresja genu MAGOHB wykazuje istotną ujemną korelację z wiekiem pacjenta
po uwzględnieniu diagnozy (β = −0.0073, 95% CI: [−0.0124, −0.0023], t = −2.856, p = 4.55×10⁻³, R² = 0.111).
Gen został wykluczony z dalszej analizy przez `CovariatesBiasReductor`.

---

## Zestawienie

| Gen    | Ensembl ID      | β       | 95% CI             | t      | p-value   | R²    |
|--------|-----------------|---------|--------------------|--------|-----------|-------|
| SCN1B  | ENSG00000105711 | +0.0131 | [0.0085, 0.0177]   | 5.608  | 4.24×10⁻⁸ | 0.148 |
| MAGOHB | ENSG00000111196 | −0.0073 | [−0.0124, −0.0023] | −2.856 | 4.55×10⁻³ | 0.111 |